In [ ]:
import os
os.chdir("/content/drive/MyDrive/RAG-based-AI-Teaching-Assistant")
print("current dir :-", os.getcwd())

current dir :- /content/drive/MyDrive/RAG-based-AI-Teaching-Assistant


In [ ]:
# download embedding model from hugging face
!pip install -U FlagEmbedding
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3',
                       use_fp32=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
# take argument as text and return its vector
def create_embedding(text):
  embeddings = model.encode(text)['dense_vecs']
  return embeddings

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import json
import joblib
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
jsons = os.listdir("jsons")
my_dicts = []
chunks_id = 0
for json_file in jsons:
  with open(f"jsons/{json_file}","r")  as f:
    content = json.load(f)
  embeddings = create_embedding([c["text"] for c in content["chunks"]])
  print(f"Creating embedding for {json_file} ",end="")
  for i, chunk in enumerate(content["chunks"]):
    chunk["chunks_id"] = chunks_id
    chunks_id+=1
    chunk["embedding"] = embeddings[i]
    my_dicts.append(chunk)
    if(i==5): # Read 5 chunks for now
      break
  break
df = pd.DataFrame.from_records(my_dicts)

Inference Embeddings: 100%|██████████| 3/3 [00:00<00:00,  4.24it/s]

Creating embedding for 2_heading_paragraphs_and_links.mp3.json 

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   number     6 non-null      object 
 1   title      6 non-null      object 
 2   start      6 non-null      float64
 3   end        6 non-null      float64
 4   text       6 non-null      object 
 5   chunks_id  6 non-null      int64  
 6   embedding  6 non-null      object 
dtypes: float64(2), int64(1), object(4)
memory usage: 468.0+ bytes


In [ ]:
df.head(6)

,number,title,start,end,text,chunks_id,embedding
0,2,heading_paragraphs_and_links,0.00,3.20,"In today's video, I will tell you about headi...",0,"[-0.03574, 0.01277, -0.01043, 0.03735, -0.0352..."
1,2,heading_paragraphs_and_links,3.20,6.36,"Along with that, I will also tell you about a...",1,"[-0.07, 0.01678, -0.0245, 0.02563, -0.0358, -0..."
2,2,heading_paragraphs_and_links,6.36,8.32,"If you have found this course awesome till now,",2,"[-0.03506, 0.02206, -0.02187, -0.01135, -0.025..."
3,2,heading_paragraphs_and_links,8.32,9.84,then do like this video.,3,"[-0.02023, 0.0109, -0.04144, 0.04034, -0.01724..."
4,2,heading_paragraphs_and_links,9.84,12.28,Your likes will help me in bringing such free...,4,"[-0.03995, -0.01906, -0.0447, 0.03012, -0.0057..."
5,2,heading_paragraphs_and_links,12.28,15.60,"In this video, we will make a small project w...",5,"[-0.03488, 0.03348, -0.00991, 0.0226, -0.01662..."


In [ ]:
incoming_query = input("Ask a question :- ")
question_embdding = create_embedding([incoming_query])[0]
print(question_embdding)

Ask a question :- free courses
[-0.02437   0.005157 -0.03702  ... -0.0389   -0.0267    0.003124]


In [ ]:
# cosine similarities

print(np.vstack(df['embedding'].values))
print(np.vstack(df["embedding"].values).shape)

[[-0.03574   0.01277  -0.01043  ...  0.02118   0.002075 -0.007183]
 [-0.07      0.01678  -0.0245   ...  0.005493 -0.02107  -0.01883 ]
 [-0.03506   0.02206  -0.02187  ...  0.001607 -0.002714  0.02495 ]
 [-0.02023   0.0109   -0.04144  ... -0.0046   -0.00962   0.02351 ]
 [-0.03995  -0.01906  -0.0447   ... -0.01859  -0.0319    0.00461 ]
 [-0.03488   0.03348  -0.00991  ...  0.01862  -0.02734  -0.01991 ]]
(6, 1024)


In [ ]:
similarities = cosine_similarity(np.vstack(df['embedding']),[question_embdding]).flatten()
print(similarities)
print(similarities.argsort())
# output
#       0          1          2          3          4          5
# [0.40594221 0.39610328 0.47558571 0.43529587 0.72489556 0.39029657]
# [5 1 0 3 2 4] (min:- 5th index max:- 4th index)

top_three_max_idx = similarities.argsort()[::-1][0:3]
print(f"Top three max indexes are {top_three_max_idx}")

[0.40594221 0.39610328 0.47558571 0.43529587 0.72489556 0.39029657]
[5 1 0 3 2 4]
Top three max indexes are [4 2 3]


In [ ]:
# getting data based on cosine similarity
new_df = df.loc[top_three_max_idx]
print(new_df[["number","title","text"]])

  number                         title  \
4      2  heading_paragraphs_and_links   
2      2  heading_paragraphs_and_links   
3      2  heading_paragraphs_and_links   

                                                text  
4   Your likes will help me in bringing such free...  
2    If you have found this course awesome till now,  
3                           then do like this video.  


In [9]:
# Store data using joblib
jsons = os.listdir("jsons")
my_dicts = []
chunks_id = 0
for json_file in jsons:
  with open(f"jsons/{json_file}","r")  as f:
    content = json.load(f)
  embeddings = create_embedding([c["text"] for c in content["chunks"]])
  print(f"Creating embedding for {json_file} ",end="")
  for i, chunk in enumerate(content["chunks"]):
    chunk["chunks_id"] = chunks_id
    chunks_id+=1
    chunk["embedding"] = embeddings[i]
    my_dicts.append(chunk)
  print("=> done")
df = pd.DataFrame.from_records(my_dicts)



pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 42.05it/s]


Inference Embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Inference Embeddings:  33%|███▎      | 1/3 [01:18<02:37, 78.61s/it]

Inference Embeddings:  67%|██████▋   | 2/3 [01:49<00:50, 50.26s/it]

Inference Embeddings: 100%|██████████| 3/3 [02:12<00:00, 44.01s/it]


Creating embedding for 2_heading_paragraphs_and_links.mp3.json => done




pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 84.71it/s]


Inference Embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Inference Embeddings:  33%|███▎      | 1/3 [00:50<01:41, 50.84s/it]

Inference Embeddings:  67%|██████▋   | 2/3 [01:20<00:38, 38.25s/it]

Inference Embeddings: 100%|██████████| 3/3 [01:41<00:00, 33.93s/it]


Creating embedding for 3_image_lists_and_tables.mp3.json => done




pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 96.53it/s]


Inference Embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Inference Embeddings:  50%|█████     | 1/2 [01:10<01:10, 70.60s/it]

Inference Embeddings: 100%|██████████| 2/2 [01:31<00:00, 45.98s/it]


Creating embedding for 4_seo_and_core_web_vitals.mp3.json => done
Creating embedding for 5_forms_and_input_tags.mp3.json => done




pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 106.36it/s]


Inference Embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Inference Embeddings:  50%|█████     | 1/2 [01:29<01:29, 89.29s/it]

Inference Embeddings: 100%|██████████| 2/2 [01:35<00:00, 47.77s/it]

Creating embedding for 1_basic_structure_of_an_html_website.mp3.json => done


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2316 entries, 0 to 2315
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   number     2316 non-null   object 
 1   title      2316 non-null   object 
 2   start      2316 non-null   float64
 3   end        2316 non-null   float64
 4   text       2316 non-null   object 
 5   chunks_id  2316 non-null   int64  
 6   embedding  2316 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 126.8+ KB


In [11]:
# Persist the embedding data
joblib.dump(df, 'embedding_data.joblib')

['embedding_data.joblib']

In [12]:
# test
new_df = joblib.load('embedding_data.joblib')
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2316 entries, 0 to 2315
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   number     2316 non-null   object 
 1   title      2316 non-null   object 
 2   start      2316 non-null   float64
 3   end        2316 non-null   float64
 4   text       2316 non-null   object 
 5   chunks_id  2316 non-null   int64  
 6   embedding  2316 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 126.8+ KB
